# EDA 03 — Uso del recurso box y citaciones

Segundo eje del reto: capacidad instalada (`agenda_box`), uso real y asistencia (`citaciones`).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from carga import cargar, base_paciente, FECHA_REFERENCIA, ORDEN_PRIORIDAD, RANGOS_NORMALES

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 30)
COLOR_PRIO = {"alta": "#c0392b", "media": "#e67e22", "baja": "#2e86c1"}

d = cargar()
pac = base_paciente(d)
lista, eventos, mediciones, agenda, citas = d["lista"], d["eventos"], d["mediciones"], d["agenda"], d["citas"]

In [ ]:
agenda["ocupacion"] = agenda["cupos_usados"] / agenda["capacidad_cupos"]
agenda["cupos_libres"] = agenda["capacidad_cupos"] - agenda["cupos_usados"]
agenda["dia_semana"] = agenda["fecha"].dt.day_name()
DIAS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
print(f"{len(agenda)} bloques · {agenda.id_box.nunique()} box · {agenda.fecha.nunique()} días hábiles · {agenda.fecha.min().date()} → {agenda.fecha.max().date()}")
print(f"Capacidad total {agenda.capacidad_cupos.sum():,} cupos · usados {agenda.cupos_usados.sum():,} · ocupación global {agenda.cupos_usados.sum() / agenda.capacidad_cupos.sum():.1%}")

## 1. Capacidad y ocupación

In [ ]:
por_esp = agenda.groupby("especialidad").agg(
    bloques=("id_box", "size"), box_distintos=("id_box", "nunique"),
    capacidad=("capacidad_cupos", "sum"), usados=("cupos_usados", "sum"), libres=("cupos_libres", "sum"),
    sobretiempo_medio=("minutos_sobretiempo", "mean"), sobretiempo_p90=("minutos_sobretiempo", lambda s: s.quantile(.9)))
por_esp["ocupacion_%"] = (por_esp["usados"] / por_esp["capacidad"] * 100).round(1)
por_esp.round(1).sort_values("ocupacion_%")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
sns.histplot(agenda, x="ocupacion", bins=20, ax=ax[0]).set_title("Ocupación por bloque")
sns.boxplot(agenda, x="ocupacion", y="especialidad", ax=ax[1]).set_title("Ocupación por especialidad")
sns.boxplot(agenda, x="minutos_sobretiempo", y="especialidad", ax=ax[2]).set_title("Sobretiempo (min) por especialidad")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
sem = agenda.assign(semana=agenda["fecha"].dt.to_period("W").dt.start_time.dt.strftime("%m-%d")).groupby(["semana", "especialidad"])[["cupos_usados", "capacidad_cupos"]].sum()
(sem["cupos_usados"] / sem["capacidad_cupos"]).unstack(1).plot(ax=ax[0], rot=60).set_title("Ocupación semanal por especialidad (semana que empieza)")
ax[0].legend(fontsize=7)
piv = agenda.pivot_table(index="especialidad", columns="dia_semana", values="ocupacion", aggfunc="mean")[DIAS]
sns.heatmap(piv, annot=True, fmt=".0%", cmap="RdYlGn_r", ax=ax[1]).set_title("Ocupación media por día de la semana")
plt.tight_layout()

In [ ]:
# Box: cuántas especialidades rota cada box y su ocupación
box = agenda.groupby("id_box").agg(dias=("fecha", "nunique"), especialidades=("especialidad", "nunique"),
                                   ocupacion=("ocupacion", "mean"), sobretiempo=("minutos_sobretiempo", "mean"))
box.round(2).T

## 2. ¿El sobretiempo se explica por la ocupación?

Hipótesis: más cupos usados → más atraso.

In [ ]:
from scipy.stats import spearmanr
r, p = spearmanr(agenda["cupos_usados"], agenda["minutos_sobretiempo"])
r2, p2 = spearmanr(agenda["ocupacion"], agenda["minutos_sobretiempo"])
print(f"Spearman cupos_usados vs sobretiempo: ρ={r:.2f} (p={p:.3g}) · ocupación vs sobretiempo: ρ={r2:.2f} (p={p2:.3g})")
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(agenda, x="cupos_usados", y="minutos_sobretiempo", color="#aed6f1", ax=ax).set_title("Sobretiempo según cupos usados")

## 3. Citaciones y asistencia

In [ ]:
ci = citas.merge(pac[["id_paciente", "edad", "sexo", "comuna", "prevision", "nivel_socioeconomico", "prioridad_inicial", "estado", "fecha_ingreso_lista"]], on="id_paciente")
ci["no_show"] = ci["estado_cita"] == "no_asistio"
ci["perdida"] = ci["estado_cita"].isin(["no_asistio", "cancelo"])
print(ci["estado_cita"].value_counts().to_string())
print(f"\nTasa no-show {ci.no_show.mean():.1%} · cupos perdidos (no-show + canceló) {ci.perdida.sum()} · pacientes citados {ci.id_paciente.nunique()}")

In [ ]:
pd.crosstab(ci["especialidad"], ci["estado_cita"], margins=True)

In [ ]:
from scipy.stats import chi2_contingency
filas = []
ci["grupo_edad"] = pd.cut(ci["edad"], [0, 40, 60, 75, 120], labels=["<40", "40-59", "60-74", "75+"])
for var in ["grupo_edad", "sexo", "comuna", "prevision", "nivel_socioeconomico", "prioridad_inicial", "especialidad"]:
    t = pd.crosstab(ci[var], ci["no_show"])
    chi2, p, *_ = chi2_contingency(t)
    filas.append({"variable": var, "p": round(p, 4), "min %": round(ci.groupby(var, observed=True)["no_show"].mean().min() * 100, 1),
                  "max %": round(ci.groupby(var, observed=True)["no_show"].mean().max() * 100, 1)})
pd.DataFrame(filas).set_index("variable")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for a, var in zip(ax, ["grupo_edad", "nivel_socioeconomico", "comuna"]):
    ci.groupby(var, observed=True)["no_show"].mean().mul(100).sort_values().plot.barh(ax=a, color="#c0392b").set_title(f"% no-show por {var}")
plt.tight_layout()

In [ ]:
# Recurrencia: ¿quien falta una vez vuelve a faltar?
ci_ord = ci.sort_values(["id_paciente", "fecha_cita"])
ci_ord["n_cita"] = ci_ord.groupby("id_paciente").cumcount() + 1
ci_ord["no_show_previo"] = ci_ord.groupby("id_paciente")["no_show"].transform(lambda s: s.shift().fillna(False).cummax())
print(ci.groupby("id_paciente").size().value_counts().sort_index().rename("pacientes por n° de citas").to_string())
ci_ord[ci_ord["n_cita"] > 1].groupby("no_show_previo")["no_show"].agg(["size", "mean"]).rename(columns={"size": "citas", "mean": "tasa_no_show"}).round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(ci.dropna(subset=["minutos_atencion_real"]), x="minutos_atencion_real", hue="especialidad", multiple="stack", bins=20, ax=ax[0]).set_title("Duración real de la atención")
ci["dias_ingreso_a_cita"] = (ci["fecha_cita"] - ci["fecha_ingreso_lista"]).dt.days
sns.boxplot(ci, x="dias_ingreso_a_cita", y="estado_cita", ax=ax[1]).set_title("Días desde el ingreso hasta la cita, según resultado")
plt.tight_layout()
ci.groupby("especialidad")["minutos_atencion_real"].describe().round(1)

In [ ]:
# Duración teórica del cupo = 300 min del bloque / capacidad
cupo_teorico = (300 / agenda["capacidad_cupos"]).describe().round(1)
print("Minutos por cupo teórico:", cupo_teorico[["min", "50%", "max"]].to_dict())
print("Minutos reales (mediana):", ci["minutos_atencion_real"].median())

## 4. Cruce citaciones ↔ agenda_box

Advertencia de `docs/datos.md`: muchas citas no tienen un bloque correspondiente.

In [ ]:
k = agenda[["id_box", "fecha", "especialidad"]].rename(columns={"fecha": "fecha_cita", "especialidad": "esp_box"})
cr = ci.merge(k, on=["id_box", "fecha_cita"], how="left")
cr["cruce"] = np.select([cr["esp_box"].isna() & (cr["fecha_cita"] < agenda["fecha"].min()),
                         cr["esp_box"].isna(),
                         cr["esp_box"] != cr["especialidad"]],
                        ["antes de la agenda", "sin bloque ese día", "box de otra especialidad"], "coincide")
cr["cruce"].value_counts()

## 5. Oferta vs. demanda por especialidad

Cupos libres en la agenda frente a pacientes en espera. Indica si el cuello de botella es capacidad o asignación.

In [ ]:
en_espera = pac[pac["estado"] == "en_espera"]
dem = en_espera.groupby("especialidad").agg(en_espera=("id_paciente", "size"),
                                            alta=("prioridad_inicial", lambda s: (s == "alta").sum()))
graves = eventos[eventos["severidad"] == "grave"]["id_paciente"].unique()
dem["con_evento_grave"] = en_espera[en_espera["id_paciente"].isin(graves)].groupby("especialidad").size()
ult_mes = agenda[agenda["fecha"] > FECHA_REFERENCIA - pd.Timedelta(days=30)]
od = dem.join(pd.DataFrame({
    "box": agenda.groupby("especialidad")["id_box"].nunique(),
    "cupos_libres_ult_30d": ult_mes.groupby("especialidad")["cupos_libres"].sum(),
})).fillna(0)
# La lista es una muestra de 500 pacientes y la agenda es la del hospital: comparar proporciones, no absolutos
od["% de la demanda"] = (od["en_espera"] / od["en_espera"].sum() * 100).round(1)
od["% de la capacidad"] = (agenda.groupby("especialidad")["capacidad_cupos"].sum() / agenda["capacidad_cupos"].sum() * 100).round(1)
od["demanda / capacidad"] = (od["% de la demanda"] / od["% de la capacidad"]).round(2)
od.sort_values("demanda / capacidad", ascending=False)

In [ ]:
ax = od[["% de la demanda", "% de la capacidad"]].sort_values("% de la demanda").plot.barh(figsize=(10, 4.5), color=["#c0392b", "#2e86c1"])
ax.set_title("Participación de cada especialidad en la demanda (en espera) y en la capacidad de box")
ax.set_xlabel("%")

## Hallazgos

1. **La capacidad está mal repartida respecto de la demanda.** Traumatología concentra el 22 % de los pacientes en espera con el 5 % de la capacidad (1 box); Cardiología, el 21 % con el 5 % (1 box). Broncopulmonar tiene el 30 % de la capacidad (6 box) para el 10 % de la demanda. Es el hallazgo de box con más impacto potencial.
2. **Ocupación pareja y moderada:** cerca de 74 % en todas las especialidades, box y días de la semana. Hay cupos libres, sobre todo donde sobra capacidad.
3. **El sobretiempo no se explica por la ocupación** (Spearman ρ = 0,02). Pero la atención real dura más que el cupo teórico (mediana 36 min frente a 30 min por cupo), lo que sugiere que el atraso viene de la duración de la atención, no del volumen.
4. **No-show de 23 %** (60 de 257 citas) y 64 cupos perdidos en total. Ninguna variable del paciente (edad, sexo, comuna, previsión, nivel socioeconómico, prioridad, especialidad) se asocia con la inasistencia (todas p > 0,4). Con 1 cita por paciente en el 95 % de los casos, no hay historial suficiente para predecir el no-show individual.
5. **Cada box atiende una sola especialidad** en todo el período.
6. **`citaciones` y `agenda_box` casi no cruzan:** solo 18 de 257 citas (7 %) caen en un bloque del mismo box, día y especialidad. Analizarlas por separado.
